In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

# Load data
df = pd.read_csv("football_data_clean.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Filter Premier League
df = df[df["league"] == "Premier League"].reset_index(drop=True)

# Encode teams
team_encoder = LabelEncoder()
all_teams = pd.concat([df["home_team_name"], df["away_team_name.x"]])
team_encoder.fit(all_teams)
df["home_team_name"] = team_encoder.transform(df["home_team_name"])
df["away_team_name.x"] = team_encoder.transform(df["away_team_name.x"])

# Encode target
y_encoded = LabelEncoder().fit_transform(df["result"])

# Features
features = [
    "home_team_name", "away_team_name.x", "home_away_flag",
    "PPG_home_last_5", "PPG_away_last_5", "average_goals_per_match_pre_match",
    "home_team_shots", "away_team_shots", "home_team_shots_on_target", "away_team_shots_on_target",
    "home_team_possession", "away_team_possession", "xG_home_team", "xG_away_team",
    "conversion_rate_home", "conversion_rate_away"
]

X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sequence preparation
seq_length = 5
predictions, actuals = [], []

initial_training_size = int(len(df) * 0.3)

# Rolling prediction
for i in tqdm(range(initial_training_size, len(df) - 1), desc="Rolling Predictions"):
    # Prepare sequences for current training
    X_seq, y_seq = [], []
    current_df = df.iloc[:i]
    current_y = y_encoded[:i]
    current_X = X_scaled[:i]

    # Create sequences separately for each team
    for team in current_df["home_team_name"].unique():
        team_df = current_df[(current_df["home_team_name"] == team) | (current_df["away_team_name.x"] == team)]
        team_X = current_X[team_df.index]
        team_y = current_y[team_df.index]

        for j in range(seq_length, len(team_df)):
            X_seq.append(team_X[j - seq_length:j])
            y_seq.append(team_y[j])

    X_seq = np.array(X_seq)
    y_seq_cat = to_categorical(y_seq, num_classes=3)

    # Build and train model
    model = Sequential([
        LSTM(64, activation='relu', input_shape=(seq_length, len(features)), return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_seq, y_seq_cat, epochs=10, batch_size=32, verbose=0)

    # Predict next match
    X_next = np.array([X_scaled[i - seq_length + 1:i + 1]])
    y_pred = np.argmax(model.predict(X_next, verbose=0), axis=1)[0]

    predictions.append(y_pred)
    actuals.append(y_encoded[i + 1])

# Evaluate
accuracy = accuracy_score(actuals, predictions)
print(f"\nRolling LSTM Model Accuracy: {accuracy:.2f}")


Rolling Predictions:   0%|          | 4/1939 [00:05<41:47,  1.30s/it]

Rolling Predictions:   0%|          | 5/1939 [00:06<41:24,  1.28s/it]

Rolling Predictions: 100%|██████████| 1939/1939 [2:13:24<00:00,  4.13s/it]   



Rolling LSTM Model Accuracy: 0.43
